# IA_agente_Camanchaca8.ipynb
## IL3.4 - Escalabilidad y Sostenibilidad
### Proyecto: Sistema Agente Camanchaca - Monitoreo Climático

Este notebook implementa estrategias de **escalabilidad y sostenibilidad** para el agente Camanchaca: cache de respuestas, enrutamiento de modelos según complejidad de la consulta, y propuestas de mejora basadas en los datos observados en los notebooks IL3.1 e IL3.2, según el indicador IE12 de la EFT (IL3.4).

**Conceptos clave aplicados:**
- Cache de respuestas (CacheLLM)
- Estimación de tokens y costos
- Enrutamiento de modelos según complejidad
- Procesamiento por lotes
- Propuestas de mejora y rediseño basadas en datos


In [ ]:
!pip install openai langchain langchain-openai langgraph requests python-dotenv -q

In [ ]:
# ============================================================
# SECCIÓN 1: CONFIGURACIÓN BASE
# ============================================================

import os
import time
import hashlib
import requests
from dataclasses import dataclass, field
from typing import List, Optional, Dict
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

load_dotenv()

if not os.getenv("OPENAI_BASE_URL"):
    raise ValueError("Falta OPENAI_BASE_URL en .env")
if not os.getenv("GITHUB_TOKEN"):
    raise ValueError("Falta GITHUB_TOKEN en .env")

llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    request_timeout=600,
    temperature=0
)

print("✓ Modelo configurado.")
print(f"Modelo: {llm.model_name}")


In [ ]:
# ============================================================
# SECCIÓN 2: HERRAMIENTAS DEL AGENTE CAMANCHACA
# (Reutilizadas de los notebooks anteriores)
# ============================================================

CENTROS = {
    "ensenada": {"lat": -41.140459, "lon": -72.404236, "nombre": "Piscicultura Petrohué"},
    "puelche":  {"lat": -41.733,    "lon": -73.602,    "nombre": "Centro Puelche"},
    "huito":    {"lat": -41.783,    "lon": -73.583,    "nombre": "Centro Huito (San José)"}
}


@tool
def get_clima_actual(centro: str) -> str:
    """Obtiene el clima actual para un centro de cultivo de Camanchaca.
    El parámetro centro puede ser: ensenada, puelche o huito."""
    if centro.lower() not in CENTROS:
        return f"Centro '{centro}' no encontrado. Opciones: ensenada, puelche, huito."

    datos = CENTROS[centro.lower()]
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={datos['lat']}&longitude={datos['lon']}"
        f"&current=temperature_2m,wind_speed_10m,precipitation,weathercode"
        f"&timezone=America/Santiago"
    )
    try:
        response = requests.get(url, timeout=10)
        data    = response.json()
        current = data["current"]
        temp      = current["temperature_2m"]
        viento    = current["wind_speed_10m"]
        lluvia    = current["precipitation"]
        codigo    = current["weathercode"]
        condicion = "Despejado" if codigo < 3 else "Nublado" if codigo < 50 else "Lluvia"
        return (
            f"Centro: {datos['nombre']}\n"
            f"Temperatura: {temp}°C\n"
            f"Viento: {viento} km/h\n"
            f"Precipitación: {lluvia} mm\n"
            f"Condición: {condicion}"
        )
    except Exception as e:
        return f"Error al obtener datos climáticos: {e}"


tools = [get_clima_actual]
agent_executor = create_react_agent(llm, tools)

print("✓ Herramientas y agente listos.")
print(f"  Herramientas: {[t.name for t in tools]}")


In [ ]:
# ============================================================
# SECCIÓN 3: CACHE DE RESPUESTAS LLM
# Reduce llamadas repetidas a GitHub Models (límite 50/día, 10/min)
# Ref: 1-scalability_sustainability.py del repositorio de la materia
# ============================================================

class CacheLLM:
    """Cache simple basado en hash del prompt para evitar llamadas repetidas."""

    def __init__(self, tamano_maximo: int = 100):
        self.cache: Dict[str, dict] = {}
        self.tamano_maximo = tamano_maximo
        self.aciertos = 0
        self.fallos   = 0

    def _generar_clave(self, prompt: str, modelo: str) -> str:
        contenido = f"{modelo}:{prompt.strip().lower()}"
        return hashlib.sha256(contenido.encode()).hexdigest()[:16]

    def obtener(self, prompt: str, modelo: str) -> Optional[str]:
        clave = self._generar_clave(prompt, modelo)
        if clave in self.cache:
            self.aciertos += 1
            return self.cache[clave]["respuesta"]
        self.fallos += 1
        return None

    def guardar(self, prompt: str, modelo: str, respuesta: str, tokens_usados: int):
        if len(self.cache) >= self.tamano_maximo:
            clave_antigua = next(iter(self.cache))
            del self.cache[clave_antigua]
        clave = self._generar_clave(prompt, modelo)
        self.cache[clave] = {"respuesta": respuesta, "tokens": tokens_usados}

    def estadisticas(self) -> dict:
        total = self.aciertos + self.fallos
        return {
            "aciertos": self.aciertos,
            "fallos": self.fallos,
            "tasa_acierto_pct": round((self.aciertos / total) * 100, 1) if total else 0,
            "entradas_en_cache": len(self.cache),
        }


cache_camanchaca = CacheLLM(tamano_maximo=50)
print("✓ CacheLLM Camanchaca inicializado (capacidad: 50 entradas).")


In [ ]:
# ============================================================
# SECCIÓN 4: ESTIMADOR DE TOKENS Y ENRUTADOR DE MODELOS
# Selecciona el modelo más eficiente según la complejidad
# de la consulta del operador
# ============================================================

def estimar_tokens(texto: str) -> int:
    """Estimación simple: ~1 token por cada 4 caracteres (aprox. para español)."""
    return max(1, len(texto) // 4)


@dataclass
class ConfigModelo:
    nombre:                  str
    costo_por_1k_tokens:     float  # USD
    latencia_promedio_ms:    float
    capacidad_maxima_tokens: int


MODELOS_DISPONIBLES = {
    "rapido":   ConfigModelo("gpt-4o-mini", 0.00015, 200,  4096),
    "estandar": ConfigModelo("gpt-4o",      0.005,   800,  8192),
    "avanzado": ConfigModelo("o1",          0.015,   2000, 16384),
}


def clasificar_complejidad(prompt: str) -> str:
    """Clasifica la complejidad de una consulta operativa de Camanchaca."""
    tokens_estimados   = estimar_tokens(prompt)
    palabras_complejas = [
        "planifica", "compara", "analiza", "evalúa toda", "calendario semanal",
        "tres centros", "estrategia", "optimiza", "reporte ejecutivo",
    ]
    indicadores = sum(1 for p in palabras_complejas if p in prompt.lower())

    if tokens_estimados > 200 or indicadores >= 2:
        return "avanzado"
    elif tokens_estimados > 50 or indicadores >= 1:
        return "estandar"
    return "rapido"


def seleccionar_modelo(prompt: str) -> ConfigModelo:
    """Selecciona el modelo más eficiente según la complejidad de la consulta."""
    nivel = clasificar_complejidad(prompt)
    return MODELOS_DISPONIBLES[nivel]


print("✓ Estimador de tokens y enrutador de modelos definidos.\n")

print("=== CLASIFICACIÓN DE CONSULTAS OPERATIVAS ===\n")
consultas_demo = [
    "¿Cuál es el clima actual en Ensenada?",
    "¿Es seguro hacer cosecha hoy en Puelche?",
    "Analiza y compara las condiciones de los tres centros, planifica el calendario semanal "
    "considerando cosecha, biometría y tratamiento, y genera un reporte ejecutivo.",
]

for c in consultas_demo:
    nivel  = clasificar_complejidad(c)
    modelo = seleccionar_modelo(c)
    print(f"Consulta: {c[:70]}...")
    print(f"  → Complejidad: {nivel} | Modelo sugerido: {modelo.nombre} "
          f"(costo: ${modelo.costo_por_1k_tokens}/1k tokens, latencia ~{modelo.latencia_promedio_ms}ms)\n")


In [ ]:
# ============================================================
# SECCIÓN 5: AGENTE CAMANCHACA CON CACHE INTEGRADO
# Evita llamadas repetidas para preguntas frecuentes
# ============================================================

def invocar_con_cache(consulta: str) -> tuple:
    """
    Invoca al agente Camanchaca usando cache de respuestas.

    Args:
        consulta: Texto de la consulta del operador.

    Returns:
        Tupla (respuesta: str, desde_cache: bool)
    """
    modelo = seleccionar_modelo(consulta)

    respuesta_cache = cache_camanchaca.obtener(consulta, modelo.nombre)
    if respuesta_cache is not None:
        return respuesta_cache, True

    response = agent_executor.invoke({"messages": [{"role": "user", "content": consulta}]})
    output   = response["messages"][-1].content

    tokens_usados = estimar_tokens(consulta) + estimar_tokens(output)
    cache_camanchaca.guardar(consulta, modelo.nombre, output, tokens_usados)

    return output, False


print("=== DEMOSTRACIÓN DE CACHE — CONSULTAS REPETIDAS ===\n")

consultas_repetidas = [
    "¿Cuál es el clima actual en Ensenada?",
    "¿Cuál es el clima actual en Puelche?",
    "¿Cuál es el clima actual en Ensenada?",  # repetida → debe venir de cache
    "¿Cuál es el clima actual en Puelche?",   # repetida → debe venir de cache
]

for i, consulta in enumerate(consultas_repetidas, 1):
    inicio = time.perf_counter()
    respuesta, desde_cache = invocar_con_cache(consulta)
    duracion_ms = (time.perf_counter() - inicio) * 1000

    origen = "📦 CACHE" if desde_cache else "🌐 API"
    print(f"{i}. [{origen}] ({duracion_ms:.1f} ms) {consulta}")
    print(f"   → {respuesta[:80]}...\n")

    if not desde_cache:
        time.sleep(7)  # Evita RateLimitError de GitHub Models (10 req/min)

print("=== ESTADÍSTICAS DEL CACHE ===")
print(cache_camanchaca.estadisticas())


In [ ]:
# ============================================================
# SECCIÓN 6: ANÁLISIS DE SOSTENIBILIDAD
# Proyección de ahorro al escalar el sistema a producción
# ============================================================

stats = cache_camanchaca.estadisticas()

print("=== PROYECCIÓN DE SOSTENIBILIDAD — AGENTE CAMANCHACA ===\n")

consultas_por_dia_sin_cache = 50  # límite diario de GitHub Models
tasa_acierto_actual = stats["tasa_acierto_pct"]
print(f"Tasa de acierto de cache observada en esta sesión: {tasa_acierto_actual}%\n")

# Si en producción ~40% de las consultas se repiten (turnos de mañana/tarde
# preguntando por los mismos centros), el cache reduce las llamadas reales:
tasa_repeticion_estimada = 0.40
llamadas_reales_estimadas = consultas_por_dia_sin_cache * (1 - tasa_repeticion_estimada)

print(f"Límite diario GitHub Models:           {consultas_por_dia_sin_cache} llamadas")
print(f"Tasa de repetición estimada (turnos):  {tasa_repeticion_estimada*100:.0f}%")
print(f"Llamadas reales estimadas con cache:   {llamadas_reales_estimadas:.0f} llamadas/día")
print(f"Capacidad operativa efectiva:          ~{consultas_por_dia_sin_cache / (1 - tasa_repeticion_estimada):.0f} "
      f"consultas de operadores/día")

print("\n=== ESCALABILIDAD POR CENTRO ===")
for centro, datos in CENTROS.items():
    print(f"  - {datos['nombre']} ({centro}): consultas cacheables por turno "
          f"(clima actual, pronóstico semanal)")


In [ ]:
# ============================================================
# SECCIÓN 7: PROPUESTAS DE MEJORA BASADAS EN DATOS OBSERVADOS
# IE12: mejoras fundamentadas en IL3.1 (métricas) e IL3.2 (trazas)
# ============================================================

propuestas = [
    {
        "area": "Reducción de latencia",
        "hallazgo": "IL3.1 mostró que la etapa de invocación del agente concentra "
                     "la mayor parte del tiempo de respuesta (>90%).",
        "propuesta": "Implementar CacheLLM (sección 3) para consultas frecuentes "
                      "de clima actual por centro, con TTL de 15 minutos.",
        "impacto": "Reduce latencia percibida de ~2-5s a <50ms en consultas repetidas."
    },
    {
        "area": "Optimización de costos / cuota",
        "hallazgo": "GitHub Models limita a 50 llamadas/día y 10/min, insuficiente "
                     "para 3 centros x múltiples turnos.",
        "propuesta": "Enrutar consultas simples (clima puntual) a modelos más "
                      "económicos (gpt-4o-mini) y reservar gpt-4o para planificación "
                      "semanal multi-centro (sección 4).",
        "impacto": "Aumenta la capacidad operativa efectiva sin exceder cuotas."
    },
    {
        "area": "Confiabilidad",
        "hallazgo": "IL3.2 identificó que mensajes vacíos generan trazas con error "
                     "sin pasar por el agente.",
        "propuesta": "Agregar validación de entrada (ya implementada en IL3.3) como "
                      "middleware obligatorio antes de cualquier invocación.",
        "impacto": "Elimina trazas de error evitables, mejorando la tasa de éxito reportada."
    },
    {
        "area": "Escalabilidad multi-centro",
        "hallazgo": "Cada centro (Ensenada, Puelche, Huito) requiere las mismas "
                     "herramientas pero con coordenadas distintas.",
        "propuesta": "Procesar consultas por lote al inicio de cada turno: "
                      "pre-calentar el cache con el clima actual de los 3 centros "
                      "en una sola ejecución batch.",
        "impacto": "Garantiza datos frescos disponibles para todos los operadores "
                    "del turno sin llamadas individuales repetidas."
    },
]

print("=== PROPUESTAS DE MEJORA — AGENTE CAMANCHACA (IE12) ===\n")
for i, p in enumerate(propuestas, 1):
    print(f"{i}. {p['area']}")
    print(f"   Hallazgo:  {p['hallazgo']}")
    print(f"   Propuesta: {p['propuesta']}")
    print(f"   Impacto:   {p['impacto']}\n")


In [ ]:
# ============================================================
# SECCIÓN 8: PROCESAMIENTO POR LOTES
# Pre-calienta el cache con el clima de los 3 centros al
# inicio del turno operativo
# ============================================================

def procesar_lote_inicio_turno(centros: list) -> list:
    """
    Pre-calienta el cache consultando el clima actual de cada centro
    al inicio del turno, optimizando las consultas posteriores de
    los operadores.

    Args:
        centros: Lista de nombres de centros a consultar.

    Returns:
        Lista de resultados por centro.
    """
    resultados = []
    for centro in centros:
        consulta = f"¿Cuál es el clima actual en {centro}?"
        inicio = time.perf_counter()
        respuesta, desde_cache = invocar_con_cache(consulta)
        duracion_ms = (time.perf_counter() - inicio) * 1000

        resultados.append({
            "centro": centro,
            "desde_cache": desde_cache,
            "duracion_ms": round(duracion_ms, 2),
            "respuesta": respuesta[:80]
        })

        if not desde_cache:
            time.sleep(7)  # Evita RateLimitError de GitHub Models (10 req/min)

    return resultados


print("=== PRE-CALENTAMIENTO DE CACHE — INICIO DE TURNO ===\n")
resultados_lote = procesar_lote_inicio_turno(list(CENTROS.keys()))

for r in resultados_lote:
    origen = "📦 CACHE" if r["desde_cache"] else "🌐 API"
    print(f"[{origen}] {r['centro']} ({r['duracion_ms']} ms): {r['respuesta']}...")

print("\n=== ESTADÍSTICAS FINALES DEL CACHE ===")
print(cache_camanchaca.estadisticas())


## Conclusión - IL3.4 / IE12

Este notebook implementó estrategias de **escalabilidad y sostenibilidad** para el agente Camanchaca, fundamentadas en los datos observados en los notebooks anteriores:

- **Cache de respuestas (CacheLLM)**: reduce llamadas repetidas a GitHub Models, crítico dado el límite de 50 llamadas/día y 10/min.
- **Enrutamiento de modelos por complejidad**: consultas puntuales (clima de un centro) pueden resolverse con modelos más económicos, reservando capacidad para análisis multi-centro.
- **Procesamiento por lotes**: pre-calienta el cache al inicio del turno operativo, asegurando datos disponibles para todos los operadores sin consultas individuales repetidas.
- **Propuestas de mejora**: cada propuesta está directamente vinculada a un hallazgo de los notebooks IL3.1 (métricas) e IL3.2 (trazas), cumpliendo con el requisito de fundamentar las mejoras en datos observados.

Estas estrategias aumentan la **escalabilidad** (soporte a más operadores y centros sin exceder cuotas de API) y la **sostenibilidad** (menor costo y latencia) del sistema agente Camanchaca.